## Load Dataset

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

In [137]:
# Load dataset
data = pd.read_csv('/kaggle/input/sms-spam-collection-dataset/spam.csv', encoding='latin-1')
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [138]:
messages = list(data['v2'])
labels = list(data['v1'])

In [139]:
y = list(pd.get_dummies(labels, drop_first=True)['spam'])

In [140]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(messages, y, test_size=0.2, random_state=0)

## Setup Transformers

In [7]:
# Install transformers
! pip install -q transformers

In [141]:
import transformers
transformers.__version__

'4.20.1'

In [142]:
import tensorflow as tf
tf.__version__

'2.6.4'

In [143]:
# Tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [144]:
train_encodings = tokenizer(x_train,
                            truncation=True,
                            padding=True)

val_encodings = tokenizer(x_test,
                            truncation=True,
                            padding=True)

In [145]:
# convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train
))
val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    y_test
))

## Load Model

In [146]:
# Load Model
from transformers import TFDistilBertForSequenceClassification

model = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_projector', 'vocab_transform', 'vocab_layer_norm', 'activation_13']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier', 'pre_classifier', 'dropout_39']
You should probably TRAIN this model on a down-stream task to be able to use i

## Train Model

In [173]:
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)

model.compile(optimizer=optimizer, loss=model.compute_loss, metrics=['accuracy'])

# start training model
model.fit(train_dataset.shuffle(100).batch(16),
          epochs=3,
          validation_data=val_dataset.shuffle(100).batch(16))

Epoch 1/3


/opt/conda/lib/python3.7/site-packages/tensorflow/python/autograph/impl/api.py:376: FutureWarning: The old compute_loss method is deprecated as it conflicts with the Keras compute_loss method added in TF 2.8. If you want the original HF compute_loss, please call hf_compute_loss() instead. From TF versions >= 2.8, or Transformers versions >= 5, calling compute_loss() will get the Keras method instead.
  return py_builtins.overload_of(f)(*args)


279/279 [==============================] - 81s 267ms/step - loss: 0.0766 - accuracy: 0.9742 - val_loss: 0.0605 - val_accuracy: 0.9874
Epoch 2/3
279/279 [==============================] - 74s 264ms/step - loss: 0.0222 - accuracy: 0.9944 - val_loss: 0.0321 - val_accuracy: 0.9928
Epoch 3/3
279/279 [==============================] - 73s 261ms/step - loss: 0.0134 - accuracy: 0.9969 - val_loss: 0.0444 - val_accuracy: 0.9910


## Evaluate model

In [175]:
model.evaluate(val_dataset.shuffle(100).batch(16))

70/70 [==============================] - 5s 70ms/step - loss: 0.0444 - accuracy: 0.9910


[0.044424645602703094, 0.9910314083099365]

In [177]:
y_pred = []
for text in x_test:
    predict_input = tokenizer.encode(text,
                                 truncation=True,
                                 padding=True,
                                 return_tensors="tf")
    
    output = model.predict(predict_input)[0]
    predictions = tf.nn.softmax(output, axis=1).numpy()
    pred = np.argmax(predictions, axis=1)
    y_pred.append(pred)

In [178]:
from sklearn.metrics import accuracy_score, classification_report

print(accuracy_score(y_pred, y_test))
print(classification_report(y_pred, y_test))

0.9910313901345291
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       959
           1       0.94      1.00      0.97       156

    accuracy                           0.99      1115
   macro avg       0.97      0.99      0.98      1115
weighted avg       0.99      0.99      0.99      1115

